# Binary X-ray fracture detector — fine-tuning

This notebook trains one reliable class: **fracture**. It does not guess the body part, so it can use verified fracture boxes from both datasets, including hand and metacarpal fractures.

Add these three Kaggle inputs and enable a GPU:

- `archive.zip.dataset` containing `BoneFractureYolo8`
- `fracAtlas.zip.dataset` containing `FracAtlas`
- A small private Kaggle dataset containing your completed `best_binary_fracture.pt` checkpoint

This run fine-tunes the completed binary checkpoint for 40 epochs at a low learning rate. The output file is `/kaggle/working/best_binary_fracture.pt`.

In [ ]:
%pip install -q "ultralytics>=8.2.0"

import random
import shutil
from collections import Counter
from pathlib import Path

import torch
import yaml
from PIL import ImageFile
from ultralytics import YOLO

ImageFile.LOAD_TRUNCATED_IMAGES = True
if not torch.cuda.is_available():
    raise RuntimeError('No GPU is attached. In Kaggle, open Settings → Accelerator → GPU, restart, then run all cells.')

SEED = 42
DEVICE = 0
random.seed(SEED)
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
DATASET = WORK / 'binary_fracture_data'
print('GPU:', torch.cuda.get_device_name(DEVICE))

def find_dataset(name, child):
    matches = [path for path in INPUT.rglob(name) if path.is_dir() and (path / child).exists()]
    return min(matches, key=lambda path: len(path.parts)) if matches else None

BONE8 = find_dataset('BoneFractureYolo8', 'train')
FRACATLAS = find_dataset('FracAtlas', 'images')
assert BONE8, 'BoneFractureYolo8 was not found in Kaggle Input.'
assert FRACATLAS, 'FracAtlas was not found in Kaggle Input.'
print('BoneFractureYolo8:', BONE8)
print('FracAtlas:', FRACATLAS)

In [ ]:
# Build a one-class detection dataset. Every retained box means fracture.
# BoneFractureYolo8 class 4 is named only 'humerus', not a confirmed fracture,
# so images containing only that ambiguous class are excluded.
if DATASET.exists():
    shutil.rmtree(DATASET)
for split in ('train', 'val', 'test'):
    (DATASET / split / 'images').mkdir(parents=True)
    (DATASET / split / 'labels').mkdir(parents=True)

FRACTURE_CLASSES = {0, 1, 2, 3, 5, 6}
AMBIGUOUS_CLASS = 4
counts = Counter()

def image_files(folder):
    return sorted(path for path in folder.iterdir() if path.suffix.lower() in {'.jpg', '.jpeg', '.png'})

def add_sample(image, lines, split, prefix):
    stem = f'{prefix}_{image.stem}'
    shutil.copy2(image, DATASET / split / 'images' / f'{stem}{image.suffix.lower()}')
    (DATASET / split / 'labels' / f'{stem}.txt').write_text('\n'.join(lines))
    counts['fracture_boxes'] += len(lines)
    counts[f'{split}_images'] += 1

def as_box(values):
    if len(values) == 4:
        x_center, y_center, width, height = values
    elif len(values) >= 6 and len(values) % 2 == 0:
        xs, ys = values[0::2], values[1::2]
        x_min, x_max = max(0.0, min(xs)), min(1.0, max(xs))
        y_min, y_max = max(0.0, min(ys)), min(1.0, max(ys))
        x_center, y_center = (x_min + x_max) / 2, (y_min + y_max) / 2
        width, height = x_max - x_min, y_max - y_min
    else:
        return None
    return (x_center, y_center, width, height) if width > 0 and height > 0 else None

def bone8_fracture_lines(label_path):
    if not label_path.exists():
        return [], False
    lines, has_ambiguous = [], False
    for row in label_path.read_text().splitlines():
        parts = row.split()
        try:
            source_class = int(parts[0])
            values = [float(value) for value in parts[1:]]
        except (IndexError, ValueError):
            continue
        if source_class == AMBIGUOUS_CLASS:
            has_ambiguous = True
            continue
        if source_class not in FRACTURE_CLASSES:
            continue
        box = as_box(values)
        if box:
            lines.append('0 ' + ' '.join(f'{value:.6f}' for value in box))
    return lines, has_ambiguous

for source_split, target_split in {'train': 'train', 'valid': 'val', 'test': 'test'}.items():
    image_dir = BONE8 / source_split / 'images'
    label_dir = BONE8 / source_split / 'labels'
    for image in image_files(image_dir):
        lines, has_ambiguous = bone8_fracture_lines(label_dir / f'{image.stem}.txt')
        if lines or not has_ambiguous:
            add_sample(image, lines, target_split, 'b8')

def fracatlas_lines(label_path):
    lines = []
    if not label_path.exists():
        return lines
    for row in label_path.read_text().splitlines():
        parts = row.split()
        try:
            box = as_box([float(value) for value in parts[1:]])
        except ValueError:
            box = None
        if box:
            lines.append('0 ' + ' '.join(f'{value:.6f}' for value in box))
    return lines

frac_labels = FRACATLAS / 'Annotations' / 'YOLO'
fractured = image_files(FRACATLAS / 'images' / 'Fractured')
normal = image_files(FRACATLAS / 'images' / 'Non_fractured')
assert fractured and normal and frac_labels.is_dir(), 'FracAtlas image or annotation folders are missing.'
# Limit normal images to twice the fractured set. This preserves negative examples
# without overwhelming the model and suppressing fracture recall.
random.shuffle(normal)
normal = normal[:min(len(normal), 2 * len(fractured))]
combined = [(image, True) for image in fractured] + [(image, False) for image in normal]
random.shuffle(combined)
train_end, val_end = int(.75 * len(combined)), int(.90 * len(combined))
for split, samples in (('train', combined[:train_end]), ('val', combined[train_end:val_end]), ('test', combined[val_end:])):
    for image, is_fractured in samples:
        lines = fracatlas_lines(frac_labels / f'{image.stem}.txt') if is_fractured else []
        if lines or not is_fractured:
            add_sample(image, lines, split, 'fa_pos' if is_fractured else 'fa_normal')

data_yaml = WORK / 'binary_fracture.yaml'
data_yaml.write_text(yaml.safe_dump({
    'path': str(DATASET), 'train': 'train/images', 'val': 'val/images',
    'test': 'test/images', 'nc': 1, 'names': ['fracture'],
}, sort_keys=False))
print('Fracture boxes:', counts['fracture_boxes'])
for split in ('train', 'val', 'test'):
    print(f'{split}: {counts[f"{split}_images"]} images')
assert counts['fracture_boxes'] > 0
print('Dataset YAML:', data_yaml)

In [ ]:
# Fine-tune the completed binary model rather than starting from YOLOv8s again.
# Put best_binary_fracture.pt in its own Kaggle Input dataset before running.
checkpoint_candidates = sorted(INPUT.rglob('best_binary_fracture.pt'), key=lambda path: len(path.parts))
assert checkpoint_candidates, (
    'Checkpoint not found. Add a Kaggle Input dataset containing best_binary_fracture.pt, then restart.'
)
checkpoint_path = checkpoint_candidates[0]
model = YOLO(str(checkpoint_path))
assert [model.names[index] for index in sorted(model.names)] == ['fracture'], (
    f'Expected the binary fracture checkpoint, got {model.names}'
)
print('Fine-tuning checkpoint:', checkpoint_path)
results = model.train(
    data=str(data_yaml), epochs=40, imgsz=640, batch=16, device=DEVICE,
    project=str(WORK / 'runs'), name='binary_fracture_finetuned', exist_ok=True,
    optimizer='AdamW', lr0=0.0001, lrf=0.1, cos_lr=True, patience=15, seed=SEED,
    mosaic=0.3, close_mosaic=5, plots=True, save=True,
)

best_path = Path(results.save_dir) / 'weights' / 'best.pt'
best = YOLO(str(best_path))
assert [best.names[index] for index in sorted(best.names)] == ['fracture']
published = WORK / 'best_binary_fracture.pt'
shutil.copy2(best_path, published)
print('Training complete.')
print('Download this Kaggle Output file:', published)